In [ ]:
import os
import sys
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')

REPO_NAME = 'MultiLexNorm_HW11'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = 'jaeiko'
GIT_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {GIT_REPO_URL}
    !cd /content/{REPO_NAME} && git checkout detection-model
else:
    !cd /content/{REPO_NAME} && git checkout detection-model && git pull origin detection-model

repo_path = f'/content/{REPO_NAME}'
os.chdir(repo_path)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

with open('requirements.txt', 'r') as f:
    reqs = f.read()
reqs = reqs.replace('pandas<2.0', 'pandas').replace('numpy<2.0', 'numpy').replace('pyarrow==14.0.2', 'pyarrow')
with open('requirements.txt', 'w') as f:
    f.write(reqs)

!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate

os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_CACHE'
login(token=userdata.get('HF_TOKEN'))
print('[System] Colab environment is ready.')


In [ ]:
import ast
import glob
import importlib
import json
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import llm
import pipeline as pipeline_module
import prompt_mfr_adapter
importlib.reload(llm)
importlib.reload(prompt_mfr_adapter)
importlib.reload(pipeline_module)

from detection import AnomalyDetector
from llm import MultilingualCorrector
from pipeline import LexicalNormalizationPipeline
from prompt_mfr_adapter import PromptMFRResources

warnings.filterwarnings('ignore')

def parse_tokens(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    return [str(token) for token in value]

def load_gemma_sentence_predictions(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [int(line.strip()) for line in f if line.strip() in {'0', '1'}]

def dump_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False)

def dump_jsonl(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

def compute_metrics(predictions, dataframe):
    tp = fp = fn = correct = total = 0
    for pred, raw_value, norm_value in zip(predictions, dataframe['raw'], dataframe['norm']):
        raw = parse_tokens(raw_value)
        norm = parse_tokens(norm_value)
        for p, r, n in zip(pred, raw, norm):
            total += 1
            correct += int(p == n)
            tp += int(r != n and p == n)
            fn += int(r != n and p != n)
            fp += int(r == n and p != r)
    err = (tp - fp) / (tp + fn) if (tp + fn) else 0.0
    return {
        'tokens': total,
        'correct': correct,
        'accuracy': correct / total if total else 0.0,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'ERR': err,
    }

def write_summary(path, experiment_name, metrics, extra):
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        f'# {experiment_name}',
        '',
        f'- fallback_strategy: `{extra["fallback_strategy"]}`',
        f'- fallback_note: `{extra["fallback_note"]}`',
        f'- Token Accuracy: `{metrics["accuracy"]:.6f}`',
        f'- ERR: `{metrics["ERR"]:.6f}`',
        f'- TP: `{metrics["TP"]}`',
        f'- FP: `{metrics["FP"]}`',
        f'- FN: `{metrics["FN"]}`',
    ]
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

print('[System] Experiment helpers loaded.')


In [ ]:
EXPERIMENT_NAME = 'prompt_mfr_mfr_suspicious_fallback_mini'
FALLBACK_STRATEGY = 'mfr_suspicious'
FALLBACK_NOTE = 'Gemma=1 and XLM-R=0 fallback uses MFR candidate_indices/prompt_hints; empty candidates means no correction.'
BATCH_SIZE = 8

local_output_dir = Path('/content') / EXPERIMENT_NAME
drive_output_dir = Path('/content/drive/MyDrive/MultiLexNorm2026/outputs') / EXPERIMENT_NAME
local_output_dir.mkdir(parents=True, exist_ok=True)
drive_output_dir.mkdir(parents=True, exist_ok=True)

model_path = '/content/drive/MyDrive/MultiLexNorm2026/models/xlmr_finetuned_colab'
detector = AnomalyDetector(model_path)
resources = PromptMFRResources('/content/MultiLexNorm_HW11/prompt_mfr_dictionary')
corrector = MultilingualCorrector('Qwen/Qwen2.5-7B-Instruct')

pipeline = LexicalNormalizationPipeline(
    detector=detector,
    dictionary=None,
    llm=corrector,
    prompt_mfr_resources=resources,
    use_prompt_mfr_dictionary=True,
    run_target_detection=False,
    fallback_strategy=FALLBACK_STRATEGY,
    record_debug=True,
)

prediction_pattern = '/content/MultiLexNorm_HW11/*/LLM-base*/experiments/exp_014_retrieval_fewshot_v1_mini_all_k20/predictions.txt'
prediction_paths = glob.glob(prediction_pattern)
if not prediction_paths:
    raise FileNotFoundError(prediction_pattern)
gemma_flags = load_gemma_sentence_predictions(prediction_paths[0])

val_path = '/content/MultiLexNorm_HW11/multilexnorm2026-dataset/mini_validation/validation_mini_1of8_seed42.parquet'
val_df = pd.read_parquet(val_path)
total_samples = len(val_df)
if len(gemma_flags) != total_samples:
    raise ValueError(f'prediction.txt rows={len(gemma_flags)}, mini validation rows={total_samples}')

final_predictions = []
for i in tqdm(range(0, total_samples, BATCH_SIZE), desc='Batch Inferencing'):
    batch_idx = range(i, min(i + BATCH_SIZE, total_samples))
    batch_raw = []
    batch_gemma = []
    batch_langs = []

    for idx in batch_idx:
        batch_raw.append(parse_tokens(val_df['raw'].iloc[idx]))
        batch_gemma.append(gemma_flags[idx])
        lang_code = val_df['language'].iloc[idx] if 'language' in val_df.columns else val_df['lang'].iloc[idx]
        batch_langs.append(str(lang_code))

    final_predictions.extend(pipeline.process_batch(batch_raw, batch_gemma, batch_langs))

    if len(final_predictions) % 500 < BATCH_SIZE:
        dump_json(local_output_dir / 'predictions_backup.json', final_predictions)
        dump_json(drive_output_dir / 'predictions_backup.json', final_predictions)

metrics = compute_metrics(final_predictions, val_df)
extra = {
    'fallback_strategy': FALLBACK_STRATEGY,
    'fallback_note': FALLBACK_NOTE,
    'validation_path': val_path,
    'prediction_path': prediction_paths[0],
}

for out_dir in (local_output_dir, drive_output_dir):
    dump_json(out_dir / 'predictions_validation_full.json', final_predictions)
    dump_json(out_dir / 'metrics.json', {**metrics, **extra})
    dump_jsonl(out_dir / 'debug.jsonl', pipeline.debug_records)
    write_summary(out_dir / 'summary.md', EXPERIMENT_NAME, metrics, extra)

print(f'[Done] Saved outputs to {local_output_dir} and {drive_output_dir}')
print(f'Accuracy={metrics["accuracy"]:.4f}, ERR={metrics["ERR"]:.4f}, TP={metrics["TP"]}, FP={metrics["FP"]}, FN={metrics["FN"]}')
